# CampusVoice-RT: 참가 시스템 S3 파인튜닝 (Colab A100)

런타임 → GPU를 **A100**으로. 흐름: 설치 → HF 인증 → 레포 → 학습용 잡음 → 학습셋 합성 → 파인튜닝 → HF Hub 업로드 → (L4에서) S3 측정.

학습 데이터는 평가셋과 겹치지 않는다. 음성은 Zeroth·FLEURS의 *train* split, 잡음은 평가셋 160클립을 제외한 FSD50K 클립, dev는 FLEURS *dev*다.

In [ ]:
!pip -q install -U qwen-asr datasets soundfile librosa silero-vad huggingface_hub
import torch; print(torch.cuda.get_device_name(0))

In [ ]:
from huggingface_hub import notebook_login; notebook_login()   # write 권한 토큰 (체크포인트 업로드)

In [ ]:
!git clone https://github.com/ekgP525/asr-challenge.git /content/asr-challenge
%cd /content/asr-challenge

## 학습용 잡음 (범주별 150클립, 평가셋 제외)

In [ ]:
!python data/scripts/build_train_set.py noise --per-category 150

## 학습셋 합성

Zeroth train 6 shard(약 2.8 GB) + FLEURS train(1.4 GB) 다운로드 후 잡음 합성. A100 VM(CPU 12코어) 기준 20~40분. 빠른 점검은 `--limit 200 --zeroth-shards 0`.

In [ ]:
!python data/scripts/build_train_set.py speech --workers 10
!tail -2 data/train/train_manifest.jsonl | cut -c1-300; wc -l data/train/train.jsonl data/train/dev/dev.jsonl

## 파인튜닝

0.6B 전체 파라미터, bf16, batch 16 × grad_acc 4, lr 1e-5, 2 epoch (약 24k 발화 → 약 750 step). 체크포인트는 `/content/ft/`에 2개만 유지. 끝나면 마지막 체크포인트를 비공개 HF 레포에 올린다(`participants/S3.json`의 `model`과 같은 이름).

In [ ]:
import os
os.environ['MODEL'] = 'Qwen/Qwen3-ASR-0.6B'
os.environ['HF_REPO'] = 'ekgP115/campusvoice-rt-s3-0.6b'   # 본인 HF 계정으로
os.environ['EPOCHS'] = '2'; os.environ['LR'] = '1e-5'
!nohup bash scripts/colab_finetune.sh > ft.log 2>&1 &
!sleep 20; tail -5 ft.log

In [ ]:
!tail -5 ft.log   # 진행 확인. 'FT_DONE'이 찍히면 완료

## dev CER 확인 (평가셋 아님)

파인튜닝 전후를 dev(FLEURS dev × 5조건)에서 비교한다. 지연 없이 Track A만 잰다.

In [ ]:
!python baseline/run_qwen3_asr.py --config participants/S3.json --manifest data/train/dev/dev_manifest.jsonl --audio-root data/train/dev --out-dir runs/dev_S3 --no-latency
!python evaluation/evaluate.py --config participants/S3.json --manifest data/train/dev/dev_manifest.jsonl --predictions runs/dev_S3/predictions.jsonl --out results/dev_S3.json --hardware 'NVIDIA A100'
!python baseline/run_qwen3_asr.py --config participants/S1b.json --manifest data/train/dev/dev_manifest.jsonl --audio-root data/train/dev --out-dir runs/dev_S1b --no-latency
!python evaluation/evaluate.py --config participants/S1b.json --manifest data/train/dev/dev_manifest.jsonl --predictions runs/dev_S1b/predictions.jsonl --out results/dev_S1b.json --hardware 'NVIDIA A100'

## 공식 측정

`reproduce_colab.ipynb`를 **L4**에서 열고 `SYSTEMS='S3'`로 실행한다(HF 토큰 필요: 비공개 모델).